# LocPick Demo: Location Choice Modeling in Python

This notebook demonstrates the core LocPick workflow:

1. **Data assembly** — Build a `ChoiceTable` from chooser and alternative data
2. **Model specification** — Define model structure with formulas and scoped terms
3. **Estimation** — Fit multinomial logit (MNL), nested logit, and mixed logit models
4. **Results** — Inspect coefficients, standard errors, and fit statistics
5. **Prediction** — Compute choice probabilities and elasticities
6. **Sampling** — Use alternative sampling for large choice sets
7. **Inference** — BHHH, robust, and clustered standard errors

In [ ]:
import numpy as np
import pandas as pd

from locpick import (
    ChoiceTable,
    EstimationProblem,
    FitDiagnostics,
    MixedLogit,
    ModelSpec,
    MultinomialLogit,
    NestedLogit,
)
from locpick.dgp import simulate_mnl
from locpick.models.mixed import ParamDistribution
from locpick.models.nested import NestingTree, NestSpec

## 1. Synthetic Data with `simulate_mnl`

LocPick includes a data generating process (DGP) utility that creates synthetic MNL choice data with known ground-truth parameters. This is useful for parameter recovery tests and demonstrations.

In [ ]:
# Generate synthetic data with known parameters
dataset = simulate_mnl(n_obs=3000, n_alts=6, seed=42)

print(f"Observations: {dataset.n_obs}")
print(f"Alternatives: {dataset.n_alts}")
print(f"True parameters: {dataset.true_params}")
print("\nChoosers (first 5 rows):")
dataset.choosers.head()

In [ ]:
print("Alternatives:")
dataset.alternatives

In [ ]:
# The DGP returns a pre-built ChoiceTable
ct = dataset.choice_table
ct

## 2. Building a ChoiceTable from Scratch

In practice, you'll assemble your own data. LocPick's `ChoiceTable.from_tables()` merges chooser attributes, alternative attributes, and chosen alternatives into a single estimation-ready object.

In [ ]:
# Create a small example from scratch
choosers = pd.DataFrame(
    {"income": [45_000, 62_000, 38_000, 55_000]},
    index=pd.Index([1, 2, 3, 4], name="oid"),
)

alternatives = pd.DataFrame(
    {
        "cost": [2.5, 3.0, 1.8, 4.2, 2.9],
        "time": [15, 22, 10, 35, 18],
    },
    index=pd.Index([10, 20, 30, 40, 50], name="aid"),
)

chosen = pd.Series([30, 10, 20, 50], index=choosers.index, name="chosen")

ct_manual = ChoiceTable.from_tables(
    choosers=choosers,
    alternatives=alternatives,
    chosen_alternatives=chosen,
)

ct_manual

### Adding interactions

You can add chooser × alternative interaction terms (e.g., `income × cost`) to capture heterogeneous preferences.

In [ ]:
# Add an interaction term (chooser × alternative)
# add_interaction_expression creates the product of two columns
ct_with_interaction = ct_manual.add_interaction_expression(
    name="income_x_cost",
    left="income",
    right="cost",
)

# The interaction column is now available
ct_with_interaction.to_frame()[["income", "cost", "income_x_cost"]].head(10)

## 3. Model Specification

LocPick uses a formula/scoped-term specification workflow:

- **Formula string**: `"alt_feature + obs_x_alt"` (R-style, via `formulaic`)
- **Scoped terms**: `ModelSpec().generic(...)`, `.alternative_specific(...)`, `.grouped(...)`

In [ ]:
# Using the DGP data for estimation
ct = dataset.choice_table

# Formula specification
spec_formula = ModelSpec(formula="alt_feature + obs_x_alt")

# Formula + generated interaction specification
spec_interaction = ModelSpec(formula="alt_feature + obs_x_alt").with_interaction(
    "obs_x_alt", "obs_feature", "alt_feature"
)

print("Formula spec:", spec_formula)
print("Formula with generated interaction:", spec_interaction)

### Alternative-specific and generic scopes

Use `.generic()` for coefficients that apply equally to all alternatives, and `.alternative_specific()` for coefficients that vary by alternative (with a reference category).

In [ ]:
# Generic: one coefficient for all alternatives
spec_generic = ModelSpec().generic("cost")

# Alternative-specific: one coefficient per alternative (except reference)
spec_as = ModelSpec().alternative_specific("cost", reference=10)

print("Generic params:", spec_generic.parameters())
print("Alt-specific params:", spec_as.parameters())

## 4. Multinomial Logit Estimation

The `MultinomialLogit` class handles estimation with either a formula or a `ModelSpec` built from formulas/scoped terms.

In [ ]:
# Estimate MNL with formula spec
model_mnl = MultinomialLogit(ct, formula="alt_feature + obs_x_alt")
result_mnl = model_mnl.fit()

# Display results
print(FitDiagnostics.summary(result_mnl))

In [ ]:
# Compare estimated parameters to ground truth
print("Parameter recovery check:")
for param, true_val in dataset.true_params.items():
    est_val = result_mnl.coefficients[param]
    pct_error = abs(est_val - true_val) / abs(true_val) * 100
    print(f"  {param:15s}: true={true_val:+.4f}  est={est_val:+.4f}  error={pct_error:.1f}%")

In [ ]:
# Access structured results
tidy = FitDiagnostics.tidy(result_mnl)
print("\nTidy results:")
tidy

In [ ]:
# Fit statistics
stats = FitDiagnostics.fit_statistics(result_mnl)
for k, v in stats.items():
    print(f"  {k}: {v}")

In [ ]:
stats

## 5. Nested Logit

When alternatives share unobserved attributes (e.g., transit modes vs. auto modes), the nested logit model accounts for correlation within nests.

In [ ]:
# Create data with a natural nest structure
n_obs = 2000
n_alts = 4
rng = np.random.default_rng(42)

choosers = pd.DataFrame(
    {"income": rng.standard_normal(n_obs)},
    index=pd.Index(np.arange(n_obs), name="oid"),
)

alternatives = pd.DataFrame(
    {
        "cost": rng.uniform(1, 10, n_alts),
        "time": rng.uniform(5, 30, n_alts),
    },
    index=pd.Index(np.arange(n_alts), name="aid"),
)

# Simulate choices using MNL
beta_cost, beta_time = -0.5, -0.1
utilities = beta_cost * alternatives["cost"].values + beta_time * alternatives["time"].values
choices = np.zeros(n_obs, dtype=int)
for i in range(n_obs):
    u = utilities + rng.gumbel(size=n_alts)
    choices[i] = np.argmax(u)

ct_nested = ChoiceTable.from_tables(
    choosers,
    alternatives,
    chosen_alternatives=pd.Series(choices, index=choosers.index),
)

# Define nest structure
tree = NestingTree(
    nests=[
        NestSpec(name="transit", alt_ids=[0, 1]),
        NestSpec(name="auto", alt_ids=[2, 3]),
    ]
)

print(f"Nests: {tree.nest_names}")
print(f"Alternatives per nest: {[len(n.alt_ids) for n in tree.nests]}")

In [ ]:
# Estimate nested logit
model_nested = NestedLogit(ct_nested, formula="cost + time", nests=tree)
result_nested = model_nested.fit()

print(FitDiagnostics.summary(result_nested))

## 6. Mixed Logit

Mixed logit (random coefficients) captures preference heterogeneity across choosers. Specify which parameters have random distributions.

In [ ]:
# Use the same data
ct_mixed = ct_nested  # reuse the nested data

# Define random coefficient distributions
# random_params maps parameter names to ParamDistribution objects
random_params = {
    "cost": ParamDistribution("normal", "cost"),  # cost coefficient ~ N(mean, sd)
}

model_mixed = MixedLogit(
    ct_mixed,
    formula="cost + time",
    random_params=random_params,
    n_draws=100,
    seed=42,
)

result_mixed = model_mixed.fit()
print(FitDiagnostics.summary(result_mixed))

## 7. Alternative Sampling

When the number of alternatives is large (e.g., thousands of census tracts), enumerating all alternatives is infeasible. LocPick supports sampling a subset of alternatives with correct likelihood correction.

In [ ]:
# Simulate a larger choice set (alternative features only, no interactions)
dataset_large = simulate_mnl(
    n_obs=500, n_alts=50, alt_params={"alt_feature": -0.5}, interaction_params={}, seed=99
)

# Sample 10 alternatives per chooser (instead of all 50)
ct_sampled = ChoiceTable.from_tables(
    dataset_large.choosers.drop(columns=["choice"]),
    dataset_large.alternatives,
    chosen_alternatives=dataset_large.choosers["choice"],
    sample_size=10,
    seed=42,
)

print(f"Full choice set: {dataset_large.n_alts} alternatives")
print(
    f"Sampled choice set: {ct_sampled.to_frame().shape[0]} rows "
    f"({ct_sampled.to_frame().shape[0] // dataset_large.n_obs} alts/obs)"
)

# The sampled ChoiceTable includes inclusion probabilities
arrays = ct_sampled.to_arrays(formula="alt_feature")
print(f"Inclusion probs shape: {arrays.inclusion_probs.shape}")
print(f"Inclusion probs (first obs, first 5 alts): {arrays.inclusion_probs[0, :5]}")

In [ ]:
# Estimate on sampled alternatives — inclusion probabilities correct the likelihood
model_sampled = MultinomialLogit(ct_sampled, formula="alt_feature")
result_sampled = model_sampled.fit()

print("\nParameter recovery with sampled alternatives:")
for param, true_val in dataset_large.true_params.items():
    est_val = result_sampled.coefficients[param]
    pct_error = abs(est_val - true_val) / abs(true_val) * 100
    print(f"  {param:15s}: true={true_val:+.4f}  est={est_val:+.4f}  error={pct_error:.1f}%")

## 8. Prediction and Elasticities

After estimation, `FitResult` provides methods for prediction, simulation, and elasticity computation.

In [ ]:
# Use the MNL model from earlier
ct = dataset.choice_table

# Choice probabilities (long-format Series indexed by obs_id, alt_id)
probs = model_mnl.probabilities(ct)
print(f"Probabilities type: {type(probs).__name__}")
print(f"Probabilities shape: {probs.shape}")
print(f"Probabilities sum to 1 per obs: {np.allclose(probs.groupby(level=0).sum(), 1.0)}")
print("\nFirst obs probabilities:")
print(probs.iloc[:6])

In [ ]:
# Systematic utilities (long-format Series)
utils = model_mnl.utilities(ct)
print(f"Utilities type: {type(utils).__name__}")
print(f"Utilities shape: {utils.shape}")
print("\nFirst obs utilities:")
print(utils.iloc[:6])

In [ ]:
# Direct elasticity: % change in choice probability for alt j
# when the variable for alt j changes by 1%
elas = model_mnl.elasticity(ct, "alt_feature")
print(f"Elasticity type: {type(elas).__name__}")
print(f"Elasticity shape: {elas.shape}")
print(f"Mean elasticity for alt_feature: {elas.mean():.4f}")

In [ ]:
# Cross elasticity: effect of changing alt_feature on other alternatives
cross_elas = model_mnl.cross_elasticity(ct, "alt_feature")
print(f"Cross elasticity type: {type(cross_elas).__name__}")
print(f"Cross elasticity shape: {cross_elas.shape}")
print(f"Mean cross elasticity: {cross_elas.mean().mean():.4f}")

In [ ]:
# Simulate choices (Monte Carlo)
sim = model_mnl.simulate(ct, n_draws=5, seed=42)
print(f"Simulation shape: {sim.shape}")
print("\nFirst 10 simulated choices:")
sim.head(10)

## 9. Inference: BHHH, Robust, and Clustered Standard Errors

Beyond the default Hessian-based standard errors, LocPick provides BHHH, robust (sandwich), and cluster-robust covariance estimators.

In [ ]:
# Default (Hessian-based) standard errors
print("Hessian-based SE:")
print(result_mnl.std_errors)

In [ ]:
# BHHH standard errors (outer product of gradients)
se_bhhh = result_mnl.std_errors_bhhh(ct)
print("\nBHHH SE:")
print(se_bhhh)

In [ ]:
# Robust (Huber-White/sandwich) standard errors
se_robust = result_mnl.std_errors_robust(ct)
print("\nRobust SE:")
print(se_robust)

In [ ]:
# Cluster-robust standard errors
# Create synthetic clusters (e.g., neighborhoods)
clusters = np.repeat(np.arange(100), 30)  # 100 clusters, 30 obs each
se_clustered = result_mnl.std_errors_clustered(ct, groups=clusters)
print("\nClustered SE (100 clusters):")
print(se_clustered)

## 10. Likelihood Ratio and Wald Tests

Compare nested models with LR tests, or test linear restrictions with Wald tests.

In [ ]:
from locpick.results import wald_test

# Fit a restricted model (only alt_feature, no interaction)
model_restricted = MultinomialLogit(ct, formula="alt_feature")
result_restricted = model_restricted.fit()

print(f"Restricted model LL: {result_restricted.log_likelihood:.2f}")
print(f"Unrestricted model LL: {result_mnl.log_likelihood:.2f}")

In [ ]:
# Likelihood ratio test: H0: obs_x_alt coefficient = 0
# Call lr_test on the restricted model, passing the unrestricted model
lr = result_restricted.lr_test(result_mnl)
print(f"\nLR test statistic: {lr.statistic:.4f}")
print(f"LR p-value: {lr.p_value:.6f}")
print(f"LR df: {lr.df}")

In [ ]:
# Wald test: H0: alt_feature = 0
# The variance-covariance matrix can be approximated from std_errors
var_cov = np.diag(result_mnl.std_errors.values**2)
wt = wald_test(
    coefficients=result_mnl.coefficients.values,
    variance_covariance=var_cov,
    r_matrix=np.array([[1.0, 0.0]]),  # test first param = 0
)
print(f"\nWald test statistic: {wt.statistic:.4f}")
print(f"Wald p-value: {wt.p_value:.6f}")

## 11. EstimationProblem

The `EstimationProblem` dataclass bundles `ChoiceArrays` with parameter metadata (initial values, bounds, fixed parameters) for a complete estimation contract.

In [ ]:
from locpick.data import EstimationProblem

# Build an EstimationProblem from a ChoiceTable + ModelSpec
spec = ModelSpec(formula="alt_feature + obs_x_alt")
problem = EstimationProblem.from_choice_table(ct, spec=spec)

print(f"Number of parameters: {problem.n_params}")
print(f"Parameter names: {problem.param_names}")
print(f"Design matrix shape: {problem.design_matrix.shape}")
print(f"Bounds: {problem.bounds}")
print(f"Fixed mask: {problem.fixed_mask}")

In [ ]:
# Estimate directly from an EstimationProblem
model_from_problem = MultinomialLogit(data=ct, problem=problem)
result_from_problem = model_from_problem.fit()

print("Estimation from EstimationProblem:")
print(FitDiagnostics.summary(result_from_problem))

## 12. Formatting Results

LocPick provides formatting utilities for publication-ready tables.

In [ ]:
from locpick import format_coefficient_table, format_fit_statistics

# Coefficient table
coef_table = format_coefficient_table(result_mnl)
print("Coefficient table:")
print(coef_table)

In [ ]:
# Fit statistics
fit_stats = format_fit_statistics(result_mnl)
print("\nFit statistics:")
print(fit_stats)

In [ ]:
# LaTeX output
latex = result_mnl.to_latex()
print("\nLaTeX output:")
print(latex[:500])  # first 500 chars

## 13. Distance Utilities

LocPick includes distance computation utilities for spatial choice models.

In [ ]:
from locpick.data.distance import (
    distance_bands,
    great_circle_distance_matrix,
    nearest_neighbors,
)

# Create synthetic coordinates
locations = pd.DataFrame(
    {
        "lat": [33.65, 33.68, 33.71, 33.64, 33.70],
        "lon": [-117.83, -117.79, -117.85, -117.90, -117.76],
    },
    index=pd.Index([100, 200, 300, 400, 500], name="zone_id"),
)

# Great-circle distance matrix (meters)
dist = great_circle_distance_matrix(locations, x="lon", y="lat")
print("Great-circle distance matrix (meters):")
print(dist.round(0))

In [ ]:
# Nearest neighbors (origins and destinations can be the same DataFrame)
nn = nearest_neighbors(locations, locations, x="lon", y="lat", k=3)
print("\n3 nearest neighbors for each location:")
print(nn)

In [ ]:
# Distance bands: which zones fall within each distance band
bands = distance_bands(dist, distances=[0, 5000, 10000, 20000])
print("\nDistance bands (0-5km, 5-10km, 10-20km):")
print(bands)

## Summary

This notebook covered the full LocPick workflow:

| Feature | Key Class/Function |
|---------|-------------------|
| Data assembly | `ChoiceTable.from_tables()` |
| Interactions | `ChoiceTable.add_interaction()` |
| Model specification | `ModelSpec` (formula + scoped terms) |
| MNL estimation | `MultinomialLogit` |
| Nested logit | `NestedLogit`, `NestingTree`, `NestSpec` |
| Mixed logit | `MixedLogit`, `ParamDistribution` |
| Alternative sampling | `ChoiceTable.from_tables(sample_size=...)` |
| Prediction | `model.probabilities()`, `model.utilities()` |
| Elasticities | `model.elasticity()`, `model.cross_elasticity()` |
| Simulation | `model.simulate()` |
| Inference | `.std_errors_bhhh()`, `.std_errors_robust()`, `.std_errors_clustered()` |
| Hypothesis tests | `FitDiagnostics.lr_test(...)`, `wald_test(...)` |
| Estimation problem | `EstimationProblem.from_choice_table()` |
| Formatting | `format_coefficient_table()`, `format_fit_statistics()` |
| Distance utilities | `great_circle_distance_matrix()`, `nearest_neighbors()` |
| DGP | `simulate_mnl()` |